# Qwen3-4B Multi-Agent Test Environment

## Setup
- Cell 1: pip install + 코드 전체 + 모델 로드 (최초 1회)
- Cell 2: 에이전트 생성 + 프롬프트 정의
- Cell 3+: 실험 (복사해서 쓰면 됨)

---

## API Reference

### Agent 생성
```python
agent = Agent("이름", "system prompt")
```

### 단일 추론
```python
# 기본 (thinking OFF, 256 tokens)
r = agent.say("질문")

# thinking ON + 토큰 늘리기
r = agent.say("질문", max_tokens=2048, thinking=True)

# 결과 접근
r["response"]   # 응답 텍스트
r["tokens"]     # 생성 토큰 수
r["time"]       # 소요 시간(초)
```

### 파라미터
| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| `max_tokens` | 256 | 생성 최대 토큰. thinking=True면 2048 권장 |
| `thinking` | False | True: Qwen3 내부 추론(`<think>`) 활성화. 느리지만 정확도↑ |

### 프롬프트 변경
```python
agent.set_prompt("새 system prompt")
```

### A → B 통신
```python
result = send(a, b, "메시지")
result["sender"]["response"]    # A의 응답
result["receiver"]["response"]  # B의 응답
result["total_tokens"]          # 총 토큰
```

### A → B → C 체인
```python
result = chain([a, b, c], "메시지")
result["final"]          # 마지막 에이전트 응답
result["total_tokens"]   # 총 토큰
result["steps"]          # 각 단계별 결과 리스트
```

### ask() 헬퍼 (ABCD 문제용)
```python
r = ask(agent, MATH_PROMPT, "질문텍스트", max_tokens=512)
r["answer"]     # 추출된 답 (A/B/C/D 또는 N/A)
r["response"]   # 전체 응답
r["tokens"]     # 토큰 수
r["time"]       # 소요 시간
```

### 유틸리티
```python
extract_number("답은 42입니다")   # → 42.0
grade(got, expected)              # 채점 (10% 오차 허용)
```

In [1]:
# ============================================================
# Cell 1: 모델 로드 (최초 1회)
# ============================================================
!pip install -q torch transformers accelerate datasets

import torch
import time
import re

# Global model/tokenizer
_model = None
_tokenizer = None
_device = None

def load_model(model_id: str = "Qwen/Qwen3-4B"):
    global _model, _tokenizer, _device
    from transformers import AutoModelForCausalLM, AutoTokenizer
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if _device == "cuda" else torch.float32
    print(f"Device: {_device}")
    if _device == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Loading {model_id}...")
    t0 = time.time()
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype,
        device_map="auto" if _device == "cuda" else None,
    )
    if _device == "cpu":
        _model = _model.to(_device)
    params = sum(p.numel() for p in _model.parameters()) / 1e9
    print(f"Loaded in {time.time()-t0:.1f}s ({params:.1f}B params)")

class Agent:
    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system_prompt = system_prompt
        self.history = []

    def say(self, message: str, max_tokens: int = 256, thinking: bool = False) -> dict:
        if _model is None:
            raise RuntimeError("load_model()을 먼저 실행하세요.")
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": message},
        ]
        text = _tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=thinking
        )
        inputs = _tokenizer(text, return_tensors="pt").to(_model.device)
        t0 = time.time()
        with torch.no_grad():
            output = _model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        elapsed = time.time() - t0
        response = _tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        gen_tokens = output.shape[1] - inputs["input_ids"].shape[1]
        result = {"response": response, "tokens": gen_tokens, "time": round(elapsed, 1)}
        self.history.append({"input": message, **result})
        return result

    def set_prompt(self, new_prompt: str):
        self.system_prompt = new_prompt
        return self

    def __repr__(self):
        return f"Agent('{self.name}')"

def send(sender, receiver, message, max_tokens=256, verbose=True):
    s = sender.say(message, max_tokens=max_tokens)
    r = receiver.say(s["response"], max_tokens=max_tokens)
    if verbose:
        print(f"[{sender.name}] {s['tokens']}tok, {s['time']}s")
        print(f"  >> {s['response'][:200]}")
        print(f"[{receiver.name}] {r['tokens']}tok, {r['time']}s")
        print(f"  >> {r['response'][:200]}")
    return {"sender": s, "receiver": r, "tx_tokens": s["tokens"], "total_tokens": s["tokens"] + r["tokens"]}

def chain(agents, message, max_tokens=256, verbose=True):
    current = message
    results = []
    for agent in agents:
        r = agent.say(current, max_tokens=max_tokens)
        results.append({"agent": agent.name, **r})
        if verbose:
            print(f"[{agent.name}] {r['tokens']}tok, {r['time']}s")
            print(f"  >> {r['response'][:200]}")
        current = r["response"]
    return {"steps": results, "final": results[-1]["response"], "total_tokens": sum(r["tokens"] for r in results)}

def extract_number(text):
    nums = re.findall(r'-?[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[0]) if nums else -999

def grade(got, expected, tolerance=0.1):
    if expected == 0:
        return abs(got) < tolerance
    return abs(got - expected) / abs(expected) < tolerance

load_model("Qwen/Qwen3-4B")

Device: cuda
GPU: Tesla T4
Loading Qwen/Qwen3-4B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded in 138.4s (4.0B params)


In [2]:
# ============================================================
# Cell 2: Summarizer-Answerer Agents + RACE 로더
# ============================================================
import random
import re as _re
from datasets import load_dataset

# --- Agent roles ---
summarizer = Agent("Summarizer", "You are a Summarizer agent.")
answerer = Agent("Answerer", "You are an Answerer agent.")

# --- RACE 로드 ---
print("Loading RACE...")
race = load_dataset("ehovy/race", "high", split="test")

def format_race(ex):
    choices = ex["options"]
    choices_text = "\n".join(f"{chr(65+i)}) {c}" for i, c in enumerate(choices))
    return {
        "passage": ex["article"],
        "question": ex["question"],
        "choices_text": choices_text,
        "answer": ex["answer"],  # "A", "B", "C", "D"
    }

def sample_race(n, seed=42):
    random.seed(seed)
    samples = random.sample(list(race), min(n, len(race)))
    return [format_race(s) for s in samples]

def extract_answer(response):
    text = response.strip()
    if text.upper() in ("A", "B", "C", "D"):
        return text.upper()
    m = _re.search(r'\\boxed\{([A-D])\}', text)
    if m: return m.group(1)
    m = _re.search(r'(?:answer|choice)[\s:is]*([A-D])\b', text, _re.IGNORECASE)
    if m: return m.group(1).upper()
    m = _re.search(r'^([A-D])[)\.]', text, _re.MULTILINE)
    if m: return m.group(1)
    m = _re.search(r'\b([A-D])\b', text)
    if m: return m.group(1)
    return "N/A"

print(f"RACE loaded: {len(race)} questions")
print(f"Agents: {summarizer}, {answerer}")

Loading RACE...


README.md: 0.00B [00:00, ?B/s]

high/test-00000-of-00001.parquet:   0%|          | 0.00/1.68M [00:00<?, ?B/s]

high/train-00000-of-00001.parquet:   0%|          | 0.00/30.4M [00:00<?, ?B/s]

high/validation-00000-of-00001.parquet:   0%|          | 0.00/1.66M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/3498 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/62445 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3451 [00:00<?, ? examples/s]

RACE loaded: 3498 questions
Agents: Agent('Summarizer'), Agent('Answerer')


---
# 핵심 아이디어 1: 상호 인지 기반 의미 통신 (RACE 20문제)
- **A (Summarizer)**: 지문 보고 요약 (항상 핵심을 첫 문장에)
- **B (Answerer)**: 질문+선택지+요약으로 답 (지문 못 봄)
- a_aware: A가 선택지를 봄 → 구분하는 핵심을 첫 문장에 (인코딩 적응)
- b_aware: B가 "A가 첫 문장에 핵심을 넣는다"를 앎 → 첫 문장에 집중 (디코딩 적응)

| 조건 | A가 보는 것 | B가 아는 것 |
|------|-----------|-----------|
| blind | 지문만 | 없음 |
| a_aware | 지문 + 선택지 | 없음 |
| b_aware | 지문만 | "A가 첫 문장에 핵심 넣음" |
| mutual | 지문 + 선택지 | "A가 첫 문장에 핵심 넣음" |

- 토큰 예산: 16 / 32 / 48 / 64

In [3]:
# ============================================================
# 핵심 아이디어 1: Tx 인지 수준별 통신 효율 (RACE 20문제)
# "중요 정보 먼저" 프롬프트 버전 — 16/32/48/64tok 풀 커브
# ============================================================

# --- A prompts ---
A_BLIND = (
    "You are a Summarizer. Read the passage and write a concise summary "
    "in 2-3 sentences. Start with the most important fact. "
    "Capture the most important facts and events."
)

A_CHOICES = (
    "You are a Summarizer. Read the passage and write a concise summary "
    "in 2-3 sentences. The reader must choose between the answer options shown below. "
    "Start with the ONE fact that best distinguishes between the options. "
    "Do NOT indicate which option is correct."
)

A_FULL = (
    "You are a Summarizer. Read the passage and write a concise summary "
    "in 2-3 sentences. The reader needs to answer the question shown below. "
    "Start with the fact most relevant to the question. "
    "Do NOT state the answer directly."
)

# --- B prompt (same for all conditions) ---
B_PROMPT = (
    "You are an Answerer. You receive a summary of a passage. "
    "Use it to answer the question. "
    "Output ONLY a single letter: A, B, C, or D."
)

# --- 3 conditions ---
CONDITIONS = ["blind", "choices_aware", "full_aware"]

TX_BUDGETS = [128]

# Sample 20 RACE questions
Q1 = sample_race(30, seed=42)
print(f"Sampled {len(Q1)} RACE questions")
for q in Q1[:3]:
    print(f"  Q: {q['question'][:50]}... ans={q['answer']}")

def run_key1(questions):
    results = {b: {c: [] for c in CONDITIONS} for b in TX_BUDGETS}

    for qi, q in enumerate(questions):
        print(f"\n--- Q{qi+1}/{len(questions)}: {q['question'][:60]}...")
        print(f"    Expected: {q['answer']}")

        for budget in TX_BUDGETS:
            for cond in CONDITIONS:
                if cond == "blind":
                    summarizer.set_prompt(A_BLIND)
                    a_input = f"Passage:\n{q['passage']}"
                elif cond == "choices_aware":
                    summarizer.set_prompt(A_CHOICES)
                    a_input = f"Passage:\n{q['passage']}\n\nAnswer options:\n{q['choices_text']}"
                else:
                    summarizer.set_prompt(A_FULL)
                    a_input = f"Passage:\n{q['passage']}\n\nQuestion: {q['question']}\nOptions:\n{q['choices_text']}"

                a_out = summarizer.say(a_input, max_tokens=budget)
                print(f"  [A-{cond} @{budget}tok]: {a_out["response"][:150]}")

                answerer.set_prompt(B_PROMPT)
                b_msg = (
                    f"Summary:\n{a_out['response']}\n\n"
                    f"Question: {q['question']}\nChoices:\n{q['choices_text']}\n\nAnswer:"
                )
                b_out = answerer.say(b_msg, max_tokens=48)
                final_ans = extract_answer(b_out["response"])
                correct = final_ans == q["answer"]
                mark = "✓" if correct else "✗"
                print(f"  [{cond:16s}] B_raw='{b_out["response"][:60]}' → {final_ans} {mark} (expected {q["answer"]})")

                results[budget][cond].append({
                    "correct": correct,
                    "answer": final_ans,
                    "expected": q["answer"],
                    "a_tokens": a_out["tokens"],
                })

        # Log @lowest budget
        b = TX_BUDGETS[0]
        row = "  ".join(
            f"{c}: {results[b][c][-1]['answer']}"
            f"{'✓' if results[b][c][-1]['correct'] else '✗'}"
            for c in CONDITIONS
        )
        print(f"  @{b}tok: {row}")

    summarizer.set_prompt("You are a Summarizer agent.")
    answerer.set_prompt("You are an Answerer agent.")
    return results

print(f"\nRunning 핵심 아이디어 1...")
k1_results = run_key1(Q1)

# Results table
print(f"\n{'Budget':<8} {'blind':<12} {'choices':<12} {'full':<12}")
print("-" * 44)
for budget in TX_BUDGETS:
    row = f"{budget}tok  "
    for cond in CONDITIONS:
        res = k1_results[budget][cond]
        n = len(res)
        acc = sum(r["correct"] for r in res) / n
        row += f"{acc*100:>5.0f}%      "
    print(row)

# Rate-distortion curve
print(f"\n--- Rate-Distortion 커브 ---")
for cond in CONDITIONS:
    points = " → ".join(f"{b}({sum(r['correct'] for r in k1_results[b][cond])/len(k1_results[b][cond])*100:.0f}%)" for b in TX_BUDGETS)
    print(f"  {cond:<16}: {points}")

# Key findings
print(f"\n--- 핵심 발견 ---")
for budget in TX_BUDGETS:
    accs = {c: sum(r["correct"] for r in k1_results[budget][c]) / len(k1_results[budget][c]) for c in CONDITIONS}
    print(f"  @{budget}tok: blind {accs['blind']*100:.0f}% → choices {accs['choices_aware']*100:.0f}% → full {accs['full_aware']*100:.0f}%")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Sampled 30 RACE questions
  Q: Why has Chen been called "market manager"?... ans=B
  Q: Hundreds of people phoned professor Sabbatucci bec... ans=C
  Q: How does a fault form?... ans=B

Running 핵심 아이디어 1...

--- Q1/30: Why has Chen been called "market manager"?...
    Expected: B
  @128tok: blind: C✗  choices_aware: C✗  full_aware: B✓

--- Q2/30: Hundreds of people phoned professor Sabbatucci because they ...
    Expected: C
  @128tok: blind: B✗  choices_aware: B✗  full_aware: C✓

--- Q3/30: How does a fault form?...
    Expected: B
  @128tok: blind: D✗  choices_aware: C✗  full_aware: B✓

--- Q4/30: Why did the bird-catcher come back to the tortoise?...
    Expected: D
  @128tok: blind: D✓  choices_aware: A✗  full_aware: D✓

--- Q5/30: What happened to the garden when the author's father was ser...
    Expected: B
  @128tok: blind: B✓  choices_aware: B✓  full_aware: B✓

--- Q6/30: Which of the following is closest to the main idea of the pa...
    Expected: C
  @128tok: blind: C✓  choi

---
# 핵심 아이디어 2: 단계별 CoT 전환 (RACE 20문제)
- **Summarizer (A)**: 3단계 CoT로 지문 처리 (이해→추론→전달)
- **Answerer (B)**: A의 Stage 3 출력만 수신 (지문 못 봄)
- Stage 3에서만 Rx 인지를 적용하면, 추론 품질은 유지하면서 통신 효율 개선

| 조건 | Stage 1-2 | Stage 3 | B |
|------|:-:|:-:|:-:|
| all_general | 일반 | 일반 | 일반 |
| all_aware | Rx 인지 | Rx 인지 | Rx 인지 |
| tx_switch | 일반 | Rx 인지 | 일반 |
| both_switch | 일반 | Rx 인지 | Rx 인지 |

In [1]:
# ============================================================
# Key Idea 2 (clean rewrite): Separate
# 1) stages of thinking
# 2) A's information access
# 3) B's calibration about A
# ============================================================

# ------------------------------------------------------------
# Stage prompts
# - Stage = reasoning step only
# - AWARENESS (question access) is injected separately
# ------------------------------------------------------------

STAGE_GEN = [
    "Read the following passage carefully. Identify the main topic, key events, and important details.\n\n"
    "Passage:\n{passage}",

    "Based on your analysis above, determine which facts are most important. "
    "Focus on concrete details such as names, numbers, events, causes, and contrasts.\n\n"
    "{prev}",

    "Write a concise message for another agent who cannot see the passage. "
    "Start with the most important fact. Keep it to 2-3 sentences.\n\n"
    "{prev}",
]

STAGE_TASK = [
    "Read the following passage carefully. Identify the main topic, key events, and important details "
    "that would help solve a reading-comprehension problem.\n\n"
    "Passage:\n{passage}\n\n"
    "{task_info}",

    "Based on your analysis above, determine which facts are most useful for answering a reading-comprehension "
    "question. Focus on details that distinguish likely confusions: who did what, when, why, what changed, "
    "and any explicit contrasts or causes.\n\n"
    "{task_info}\n\n"
    "{prev}",

    "Write a concise message for an agent answering a reading-comprehension question. "
    "That agent cannot see the passage. Start with the fact most useful for distinguishing among plausible "
    "answer choices. Include only the most decision-relevant details. Keep it to 2-3 sentences. "
    "Do NOT state the answer directly.\n\n"
    "{task_info}\n\n"
    "{prev}",
]

# ------------------------------------------------------------
# A information access
# - aware: A sees question + choices in every stage
# - blind: A never sees them
# ------------------------------------------------------------

def build_task_info(q, a_aware):
    if not a_aware:
        return ""
    return (
        "Reading-comprehension target:\n"
        f"Question: {q['question']}\n"
        f"Choices:\n{q['choices_text']}"
    )

# ------------------------------------------------------------
# B prompts
# - blind: just answer from summary
# - aware_general: knows A used general summarization
# - aware_task_blindA: knows A used task-oriented reasoning but did NOT see the actual question
# - aware_task_awareA: knows A used task-oriented reasoning and DID see the question/choices
# ------------------------------------------------------------

B_BLIND_K2 = (
    "You are an Answerer. You receive a summary of a passage. "
    "Use it to answer the question. "
    "Output ONLY a single letter: A, B, C, or D."
)

B_AWARE_GENERAL = (
    "You are an Answerer. You receive a summary from a Summarizer agent.\n\n"
    "About the Summarizer's process:\n"
    "- The Summarizer analyzed the passage in 3 stages\n"
    "- The reasoning was general-purpose, not optimized for a specific reading-comprehension task\n"
    "- The Summarizer did NOT see the question or answer choices\n\n"
    "Interpret accordingly:\n"
    "- The summary may contain broadly important information rather than question-specific evidence\n"
    "- Some details needed for this question may be missing\n"
    "- Compare all answer choices carefully against the summary\n\n"
    "Output ONLY a single letter: A, B, C, or D."
)

B_AWARE_TASK_BLIND_A = (
    "You are an Answerer. You receive a summary from a Summarizer agent.\n\n"
    "About the Summarizer's process:\n"
    "- The Summarizer analyzed the passage in 3 stages\n"
    "- The reasoning was oriented toward reading-comprehension usefulness\n"
    "- However, the Summarizer did NOT see the exact question or answer choices\n\n"
    "Interpret accordingly:\n"
    "- The summary may emphasize details that are often useful for answer selection\n"
    "- But it is still not guaranteed to target this specific question\n"
    "- Treat emphasized details as helpful signals, not decisive proof\n"
    "- Compare all answer choices carefully against the summary\n\n"
    "Output ONLY a single letter: A, B, C, or D."
)

B_AWARE_TASK_AWARE_A = (
    "You are an Answerer. You receive a summary from a Summarizer agent.\n\n"
    "About the Summarizer's process:\n"
    "- The Summarizer analyzed the passage in 3 stages\n"
    "- The reasoning was oriented toward reading-comprehension usefulness\n"
    "- The Summarizer DID see the exact question and answer choices while composing the summary\n\n"
    "Interpret accordingly:\n"
    "- The summary was intentionally shaped to help discriminate among the answer choices\n"
    "- Still, the summary may omit some context and should not be treated as infallible\n"
    "- Use the summary as targeted evidence, then compare all answer choices carefully\n\n"
    "Output ONLY a single letter: A, B, C, or D."
)

B_PROMPTS_K2 = {
    "blind": B_BLIND_K2,
    "aware_general": B_AWARE_GENERAL,
    "aware_task_blind_a": B_AWARE_TASK_BLIND_A,
    "aware_task_aware_a": B_AWARE_TASK_AWARE_A,
}

# ------------------------------------------------------------
# Conditions
# - stage_mode: how A thinks
# - a_aware: what A knows
# - b_mode: what B knows about A
# ------------------------------------------------------------

COT_CONDITIONS = {
    # Baseline: general reasoning, A doesn't know question, B doesn't know setup
    "gen_blind__b_blind": {
        "stage_mode": "general",
        "a_aware": False,
        "b_mode": "blind",
    },

    # B is calibrated that A made a general-purpose summary
    "gen_blind__b_aware": {
        "stage_mode": "general",
        "a_aware": False,
        "b_mode": "aware_general",
    },

    # A uses task-oriented reasoning style, but still doesn't know exact question
    "task_blind__b_blind": {
        "stage_mode": "task",
        "a_aware": False,
        "b_mode": "blind",
    },

    # B knows A used task-oriented reasoning but A was still question-blind
    "task_blind__b_aware": {
        "stage_mode": "task",
        "a_aware": False,
        "b_mode": "aware_task_blind_a",
    },

    # A knows the question/choices and uses task-oriented reasoning; B unaware
    "task_aware__b_blind": {
        "stage_mode": "task",
        "a_aware": True,
        "b_mode": "blind",
    },

    # Full aligned condition: A knows question/choices, task-oriented reasoning, B calibrated
    "task_aware__b_aware": {
        "stage_mode": "task",
        "a_aware": True,
        "b_mode": "aware_task_aware_a",
    },
}

# ------------------------------------------------------------
# Budgets
# ------------------------------------------------------------

STAGE12_BUDGET = 64
TX_BUDGETS_S3 = [64]

Q2 = sample_race(30, seed=42)
print(f"Key Idea 2 (clean rewrite): {len(Q2)} RACE questions, Stage3 budgets={TX_BUDGETS_S3}")

# ------------------------------------------------------------
# Helper: run A once per unique config
# ------------------------------------------------------------

def run_summarizer_for_condition(q, stage_mode, a_aware, s3_budget):
    stages = STAGE_TASK if stage_mode == "task" else STAGE_GEN
    task_info = build_task_info(q, a_aware)

    prev = ""
    for i in range(3):
        msg = stages[i].format(
            passage=q["passage"],
            prev=prev,
            task_info=task_info,
        )
        budget = STAGE12_BUDGET if i < 2 else s3_budget
        summarizer.set_prompt("You are a Summarizer agent.")
        r = summarizer.say(msg, max_tokens=budget)
        prev = r["response"].strip()

    return prev

# ------------------------------------------------------------
# Run
# ------------------------------------------------------------

k2_results = {b: {c: [] for c in COT_CONDITIONS} for b in TX_BUDGETS_S3}

for qi, q in enumerate(Q2):
    print(f"\n--- Q{qi+1}/{len(Q2)}: {q['question'][:60]}...")
    print(f"    Expected: {q['answer']}")

    for s3_budget in TX_BUDGETS_S3:
        # Cache A outputs by (stage_mode, a_aware)
        a_cache = {}

        for cond_name, cfg in COT_CONDITIONS.items():
            a_key = (cfg["stage_mode"], cfg["a_aware"])

            if a_key not in a_cache:
                a_msg = run_summarizer_for_condition(
                    q=q,
                    stage_mode=cfg["stage_mode"],
                    a_aware=cfg["a_aware"],
                    s3_budget=s3_budget,
                )
                a_cache[a_key] = a_msg
                print(f"  [A {cond_name} @S3={s3_budget}tok]: {a_msg[:180]}")
            else:
                a_msg = a_cache[a_key]

            b_sys = B_PROMPTS_K2[cfg["b_mode"]]
            answerer.set_prompt(b_sys)
            b_out = answerer.say(
                f"Summary:\n{a_msg}\n\n"
                f"Question: {q['question']}\n"
                f"Choices:\n{q['choices_text']}\n\n"
                f"Answer:",
                max_tokens=48,
            )

            ans = extract_answer(b_out["response"])
            correct_k2 = ans == q["answer"]
            mark_k2 = "✓" if correct_k2 else "✗"
            print(f"  [{cond:14s}] B_raw='{b_out["response"][:60]}' → {ans} {mark_k2} (expected {q["answer"]})")
            correct = ans == q["answer"]

            k2_results[s3_budget][cond_name].append({
                "correct": correct,
                "answer": ans,
                "expected": q["answer"],
            })

            mark = "✓" if correct else "✗"
            print(f"  [{cond_name:20s}] -> {ans} {mark}")

# ------------------------------------------------------------
# Restore prompts
# ------------------------------------------------------------

summarizer.set_prompt("You are a Summarizer agent.")
answerer.set_prompt("You are an Answerer agent.")

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

ordered_conditions = [
    "gen_blind__b_blind",
    "gen_blind__b_aware",
    "task_blind__b_blind",
    "task_blind__b_aware",
    "task_aware__b_blind",
    "task_aware__b_aware",
]

print(
    f"\n{'S3 Budget':<10} "
    f"{'gen_bb':<10} "
    f"{'gen_ba':<10} "
    f"{'task_bb':<10} "
    f"{'task_ba':<10} "
    f"{'aware_bb':<10} "
    f"{'aware_ba':<10}"
)
print("-" * 80)

for b in TX_BUDGETS_S3:
    row = f"{str(b)+'tok':<10} "
    for c in ordered_conditions:
        res = k2_results[b][c]
        acc = sum(r["correct"] for r in res) / len(res)
        row += f"{acc*100:>5.0f}%     "
    print(row)

# ------------------------------------------------------------
# Focused comparisons
# ------------------------------------------------------------

def acc(results):
    return sum(r["correct"] for r in results) / len(results)

print("\n--- 핵심 비교 ---")
for b in TX_BUDGETS_S3:
    gen_bb   = acc(k2_results[b]["gen_blind__b_blind"])
    gen_ba   = acc(k2_results[b]["gen_blind__b_aware"])
    task_bb  = acc(k2_results[b]["task_blind__b_blind"])
    task_ba  = acc(k2_results[b]["task_blind__b_aware"])
    aware_bb = acc(k2_results[b]["task_aware__b_blind"])
    aware_ba = acc(k2_results[b]["task_aware__b_aware"])

    print(f"@S3={b}tok")
    print(f"  baseline                      : {gen_bb*100:.0f}%")
    print(f"  + B calibration on general A  : {gen_ba*100:.0f}%  ({(gen_ba-gen_bb)*100:+.1f}pt)")
    print(f"  + task-style thinking only    : {task_bb*100:.0f}%  ({(task_bb-gen_bb)*100:+.1f}pt)")
    print(f"  + task-style + B calibration  : {task_ba*100:.0f}%  ({(task_ba-gen_bb)*100:+.1f}pt)")
    print(f"  + A question access only      : {aware_bb*100:.0f}%  ({(aware_bb-gen_bb)*100:+.1f}pt)")
    print(f"  + full alignment (A+B aware)  : {aware_ba*100:.0f}%  ({(aware_ba-gen_bb)*100:+.1f}pt)")

NameError: name 'sample_race' is not defined

---
# Key Idea 4: Task Scheduling (MMLU 30문제, 도메인별 10개)
- **3 에이전트**: `Math`, `Science`, `CS` (고정, 각자 도메인 프롬프트)
- A→B→C 체인, 6순열 전부 테스트
- **첫 번째 에이전트만 문제를 보고**, 이후 에이전트는 이전 메시지만 수신
- 도메인 매칭이 중요: 물리 문제를 Science가 먼저 보면 유리

In [ ]:
# ============================================================
# Key Idea 4: MMLU 30문제 x 6순열 (Information Asymmetry Chain)
# Only the FIRST agent sees the original question.
# Each subsequent agent only receives the previous agent's message.
# ============================================================
Q4_math = sample_mmlu(10, domain="Math", seed=77)
Q4_sci  = sample_mmlu(10, domain="Science", seed=78)
Q4_cs   = sample_mmlu(10, domain="CS", seed=79)
Q4_all  = [(q, "Math") for q in Q4_math] + [(q, "Science") for q in Q4_sci] + [(q, "CS") for q in Q4_cs]
print(f"Sampled: Math={len(Q4_math)}, Science={len(Q4_sci)}, CS={len(Q4_cs)}, Total={len(Q4_all)}")

agent_names = ["Math", "Science", "CS"]
all_orders = list(permutations(agent_names))

DOMAIN_LABELS = {
    "Math": "mathematics",
    "Science": "natural science",
    "CS": "computer science",
}

def run_chain_mmlu(order, question_text):
    """3-agent chain. Only first agent sees the question. Others get previous message only."""
    prev = ""
    total_tokens = 0

    for i, name in enumerate(order):
        domain_label = DOMAIN_LABELS[name]

        if i == 0:
            # First agent: sees the question, must communicate to next
            AGENTS[name].set_prompt(
                f"You are ONLY a {domain_label} domain expert. "
                "You must analyze this problem and send a message to the next agent "
                "who CANNOT see the question. Include enough information for them to continue."
            )
            msg = f"Question: {question_text}\nAnalyze and send a message for the next agent."
        elif i == 1:
            # Middle agent: receives message, processes, forwards
            AGENTS[name].set_prompt(
                f"You are ONLY a {domain_label} domain expert. "
                "You received a message from the previous agent about a problem they analyzed. "
                "Add your expertise and forward a refined message to the next agent."
            )
            msg = f"Message from previous agent:\n{prev}\n\nRefine this analysis with your expertise and forward."
        else:
            # Last agent: receives message, gives final answer
            AGENTS[name].set_prompt(
                f"You are ONLY a {domain_label} domain expert. "
                "You received a message from previous agents about a problem. "
                "Based on their analysis, determine the final answer. "
                "Put your answer inside \\boxed{X} where X is A, B, C, or D."
            )
            msg = f"Message from previous agents:\n{prev}\n\nWhat is the correct answer? Put it inside \\boxed{{X}}."

        r = AGENTS[name].say(msg, max_tokens=128)
        prev = r["response"]
        total_tokens += r["tokens"]

    # Restore prompts
    for name in order:
        AGENTS[name].set_prompt(PROMPTS[name])

    match = _re.search(r'\\boxed\{([A-D])\}', prev)
    return match.group(1) if match else "N/A", total_tokens

# Run all permutations
k4_results = {}
for order in all_orders:
    order_key = "→".join(order)
    domain_scores = {"Math": [], "Science": [], "CS": []}
    total_tok = 0

    print(f"\n{'='*50}")
    print(f"  Order: {order_key}")
    print(f"{'='*50}")

    for q, domain in Q4_all:
        ans, tok = run_chain_mmlu(order, q["text"])
        correct = ans == q["answer"]
        domain_scores[domain].append(correct)
        total_tok += tok

    all_scores = [s for scores in domain_scores.values() for s in scores]
    k4_results[order_key] = {
        "accuracy": sum(all_scores) / len(all_scores),
        "by_domain": {d: sum(s)/len(s) for d, s in domain_scores.items()},
        "correct": sum(all_scores),
        "total": len(all_scores),
        "total_tokens": total_tok,
    }
    print(f"  Total: {sum(all_scores)}/{len(all_scores)} "
          f"(M:{sum(domain_scores['Math'])}/10 S:{sum(domain_scores['Science'])}/10 C:{sum(domain_scores['CS'])}/10)")

print("\nDone.")

In [ ]:
# ============================================================
# Key Idea 4: 결과 비교표
# ============================================================
sorted_k4 = sorted(k4_results.items(), key=lambda x: -x[1]["accuracy"])

print(f"{'Order':<20} {'Total':<10} {'Math':<8} {'Science':<8} {'CS':<8} {'Tokens'}")
print("-" * 65)
for order_key, r in sorted_k4:
    d = r["by_domain"]
    print(f"{order_key:<20} {r['accuracy']*100:>5.1f}%    "
          f"{d['Math']*100:>5.1f}%  {d['Science']*100:>5.1f}%  {d['CS']*100:>5.1f}%  {r['total_tokens']}")

print(f"\nBest:  {sorted_k4[0][0]} ({sorted_k4[0][1]['accuracy']*100:.1f}%)")
print(f"Worst: {sorted_k4[-1][0]} ({sorted_k4[-1][1]['accuracy']*100:.1f}%)")

# Domain-optimal: which order is best for each domain?
print("\nBest order per domain (first agent = encoder):")
for domain in ["Math", "Science", "CS"]:
    best = max(k4_results.items(), key=lambda x: x[1]["by_domain"][domain])
    print(f"  {domain} questions: {best[0]} ({best[1]['by_domain'][domain]*100:.0f}%)")